In [1]:
# CELL 1 - Setup (Colab already has requests, bs4, pandas, tqdm)
import re, time, random, requests, pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

START_YEAR, END_YEAR = 2020, 2026
GRANULARITY = "quarterly"   # "quarterly" = fast, "monthly" = more detail
WORKERS = 1                 # one at a time = gentlest on archive.org
ONLY = ["microsoft365", "googleone"]   # services to run now (netflix already done)

SERVICES = {
    "netflix": {
        "urls": ["https://help.netflix.com/en/node/24926",
                 "https://www.netflix.com/in/"],
        "plans": ["Mobile", "Basic", "Standard", "Premium", "with ads"],
    },
    "microsoft365": {
        "urls": ["https://www.microsoft.com/en-in/microsoft-365/p/microsoft-365-personal/cfq7ttc0k5bf",
                 "https://www.microsoft.com/en-us/microsoft-365/p/microsoft-365-personal/cfq7ttc0k5bf",
                 "https://www.microsoft.com/en-in/microsoft-365/compare-all-microsoft-365-products",
                 "https://www.microsoft.com/en-us/microsoft-365/compare-all-microsoft-365-products"],
        "plans": ["Personal", "Family", "Basic", "Business Basic",
                  "Business Standard", "Business Premium"],
    },
    "googleone": {
        "urls": ["https://one.google.com/about/plans",
                 "https://one.google.com/about/plans?hl=en-IN",
                 "https://one.google.com/about"],
        "plans": ["Basic", "Standard", "Premium", "AI Premium", "100 GB", "200 GB", "2 TB"],
    },
}

PRICE_RE = re.compile(r"(?:₹|Rs\.?|INR|US\$|\$|£|€)\s?\d[\d,]*(?:\.\d{1,2})?")
HEADERS = {"User-Agent": "Mozilla/5.0 (student research project)"}
print("Setup done")


Setup done


In [2]:
# CELL 2 - Functions
def list_snapshots(url):
    last = None
    for attempt in range(5):                       # retry with growing wait
        try:
            r = requests.get("https://web.archive.org/cdx/search/cdx", params={
                "url": url, "output": "json", "from": str(START_YEAR), "to": str(END_YEAR),
                "filter": "statuscode:200", "collapse": "timestamp:6", "fl": "timestamp,original"},
                headers=HEADERS, timeout=(15, 60))
            r.raise_for_status()
            rows = r.json()[1:]
            break
        except Exception as e:
            last = e
            wait = 20 * (attempt + 1)
            print(f"   archive.org refused, waiting {wait}s (try {attempt+1}/5)...")
            time.sleep(wait)
    else:
        raise last
    if GRANULARITY == "quarterly":
        seen, out = set(), []
        for ts, orig in rows:
            q = (ts[:4], (int(ts[4:6]) - 1) // 3)
            if q not in seen:
                seen.add(q); out.append((ts, orig))
        return out
    return [tuple(x) for x in rows]

def fetch(ts, original):
    snap_url = f"https://web.archive.org/web/{ts}id_/{original}"
    for attempt in range(2):
        try:
            r = requests.get(snap_url, headers=HEADERS, timeout=(10, 30))
            if r.status_code == 200:
                return snap_url, r.text
        except requests.RequestException:
            pass
        time.sleep(2)
    return snap_url, None

def extract_prices(html, plans):
    text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
    found = []
    for plan in plans:
        for m in re.finditer(re.escape(plan), text, flags=re.I):
            p = PRICE_RE.search(text[m.end(): m.end() + 150])
            if p:
                found.append((plan, p.group(0))); break
    return found

def work(service, page, ts, original, plans):
    time.sleep(random.uniform(3, 5))          # polite jitter
    snap_url, html = fetch(ts, original)
    if not html:
        return []
    return [{"service": service, "snapshot_date": f"{ts[:4]}-{ts[4:6]}-{ts[6:8]}",
             "plan": plan, "price_raw": price, "source_page": page, "snapshot_url": snap_url}
            for plan, price in extract_prices(html, plans)]
print("Functions ready")


Functions ready


In [3]:
# CELL 3 - Run selected services slowly, then merge every saved CSV
import glob
for service in ONLY:
    cfg = SERVICES[service]
    jobs = []
    for page in cfg["urls"]:
        try:
            snaps = list_snapshots(page)
            print(f"[{service}] {len(snaps)} snapshots for {page}")
            jobs += [(page, ts, orig) for ts, orig in snaps]
        except Exception as e:
            print(f"[{service}] gave up on {page}: {str(e)[:80]}")
        time.sleep(10)                              # pause between pages
    rows = []
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = [ex.submit(work, service, p, ts, o, cfg["plans"]) for p, ts, o in jobs]
        for f in tqdm(as_completed(futs), total=len(futs), desc=service):
            try:
                rows += f.result()
            except Exception:
                pass
    pd.DataFrame(rows).drop_duplicates().to_csv(f"prices_{service}.csv", index=False)
    print(f"[{service}] done: {len(rows)} price rows")

parts = [pd.read_csv(f) for f in glob.glob("prices_*.csv") if "all" not in f]
df = pd.concat(parts, ignore_index=True).drop_duplicates()
df["price_num"] = df["price_raw"].str.replace(r"[^\d.]", "", regex=True).astype(float)
df = df.sort_values(["service", "plan", "snapshot_date"])
df.to_csv("prices_all.csv", index=False)
print(df["service"].value_counts())
df.head(20)


[microsoft365] 15 snapshots for https://www.microsoft.com/en-in/microsoft-365/p/microsoft-365-personal/cfq7ttc0k5bf
[microsoft365] 26 snapshots for https://www.microsoft.com/en-us/microsoft-365/p/microsoft-365-personal/cfq7ttc0k5bf
[microsoft365] 3 snapshots for https://www.microsoft.com/en-in/microsoft-365/compare-all-microsoft-365-products
[microsoft365] 7 snapshots for https://www.microsoft.com/en-us/microsoft-365/compare-all-microsoft-365-products


microsoft365:   0%|          | 0/51 [00:00<?, ?it/s]

[microsoft365] done: 23 price rows
   archive.org refused, waiting 20s (try 1/5)...
   archive.org refused, waiting 40s (try 2/5)...
[googleone] 25 snapshots for https://one.google.com/about/plans
[googleone] 2 snapshots for https://one.google.com/about/plans?hl=en-IN
[googleone] 28 snapshots for https://one.google.com/about


googleone:   0%|          | 0/55 [00:00<?, ?it/s]

[googleone] done: 158 price rows
service
googleone       158
microsoft365     23
Name: count, dtype: int64


,service,snapshot_date,plan,price_raw,source_page,snapshot_url,price_num
109,googleone,2020-01-01,100 GB,€1,https://one.google.com/about,https://web.archive.org/web/20200101123436id_/...,1.00
112,googleone,2020-07-01,100 GB,$1.99,https://one.google.com/about,https://web.archive.org/web/20200701201720id_/...,1.99
23,googleone,2020-11-18,100 GB,$1.99,https://one.google.com/about/plans,https://web.archive.org/web/20201118204453id_/...,1.99
26,googleone,2021-01-06,100 GB,$1.99,https://one.google.com/about/plans,https://web.archive.org/web/20210106201306id_/...,1.99
29,googleone,2021-04-01,100 GB,$1.99,https://one.google.com/about/plans,https://web.archive.org/web/20210401023542id_/...,1.99
115,googleone,2021-04-01,100 GB,$1.99,https://one.google.com/about,https://web.archive.org/web/20210401010705id_/...,1.99
32,googleone,2021-07-01,100 GB,$1.99,https://one.google.com/about/plans,https://web.archive.org/web/20210701040134id_/...,1.99
118,googleone,2021-07-01,100 GB,€1.99,https://one.google.com/about,https://web.archive.org/web/20210701002117id_/...,1.99
124,googleone,2021-10-01,100 GB,$1.99,https://one.google.com/about,https://web.archive.org/web/20211001073639id_/...,1.99
38,googleone,2021-10-02,100 GB,$1.99,https://one.google.com/about/plans,https://web.archive.org/web/20211002014814id_/...,1.99


In [4]:
# CELL 4 - Download the results to your computer
from google.colab import files
files.download("prices_all.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# CELL 5 - Convert results to JSON (combined, per-service, and one file per snapshot date)
import os, json, shutil
import pandas as pd
from datetime import datetime, timezone

df = pd.read_csv("prices_all.csv")
out = "json_out"
shutil.rmtree(out, ignore_errors=True)
os.makedirs(out)

# 1) one combined file
df.to_json(f"{out}/prices_all.json", orient="records", indent=2)

# 2) one file per service
for service, g in df.groupby("service"):
    g.to_json(f"{out}/{service}.json", orient="records", indent=2)

# 3) one file per service + snapshot date (same shape as your Lambda's raw/<service>/<date>.json)
now = datetime.now(timezone.utc).isoformat()
for (service, date), g in df.groupby(["service", "snapshot_date"]):
    os.makedirs(f"{out}/raw/{service}", exist_ok=True)
    record = {
        "service": service,
        "snapshot_date": date,
        "scraped_at": now,
        "prices": [
            {"plan": r.plan, "price_raw": r.price_raw, "price_num": r.price_num,
             "source_page": r.source_page, "snapshot_url": r.snapshot_url}
            for r in g.itertuples()
        ],
    }
    with open(f"{out}/raw/{service}/{date}.json", "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

n = sum(len(files) for _, _, files in os.walk(out))
print(f"Created {n} JSON files in {out}/")


Created 37 JSON files in json_out/


In [6]:
# CELL 6 - Zip and download all JSON files
import shutil
from google.colab import files
shutil.make_archive("json_out", "zip", "json_out")
files.download("json_out.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>